In [ ]:
# Загрузка переменных окружения и инициализация клиента Cohere
import os
from pathlib import Path
from dotenv import load_dotenv
import sqlite3  # ← ДОБАВИТЬ: встроен в Python, устанавливать не нужно

env_file = Path(__file__).parent.parent / '.env' if '__file__' in dir() else Path('..') / '.env'
load_dotenv(env_file, override=True)
cohere_client = cohere.Client(os.getenv("COHERE_API_KEY"))
print(f"Cohere ключ: {os.getenv('COHERE_API_KEY', 'НЕ НАЙДЕН')[:8]}...")

In [ ]:
# Принудительное извлечение OpenAI API Key из .env с механизмом проверки
import os
from pathlib import Path
from dotenv import load_dotenv

# Находим .env относительно текущего файла
env_file = Path(__file__).parent.parent / '.env' if '__file__' in dir() else Path('..') / '.env'

# Принудительно перезаписываем даже если уже загружено
load_dotenv(env_file, override=True)

# Явная проверка — если не загрузилось через dotenv, читаем сами
if not os.environ.get('OPENAI_API_KEY'):
    with open(env_file) as f:
        for line in f:
            if 'OPENAI_API_KEY' in line:
                key = line.split('=', 1)[1].strip().strip('"')
                os.environ['OPENAI_API_KEY'] = key
                break

key = os.environ.get('OPENAI_API_KEY', '')
print(f"Ключ загружен: {key[:8]}...{key[-4:]}") # показываем начало и конец для проверки

In [ ]:
# --- СИСТЕМНЫЕ МОДУЛИ И УТИЛИТЫ ---
import os  #
import gc  #
import json  #
import re  #
import time  #
import glob  #
import logging  #
import shutil  #
import chardet  #
import hashlib  #
import sqlite3  #
from pathlib import Path  #
from dotenv import load_dotenv  #
from itertools import zip_longest  #
from tqdm.auto import tqdm  #

# --- ОБРАБОТКА ДАННЫХ И МЕТРИКИ ТОКЕНОВ ---
import numpy as np  #
import pandas as pd  #
import tiktoken  #

# --- ВИЗУАЛИЗАЦИЯ (TSNE / Plotly) ---
from sklearn.manifold import TSNE  #
import plotly.graph_objects as go  #

# --- DOCUMENT PARSING И NLP (Natasha) ---
from docx import Document as DocxDocument  #
from natasha import (
    Segmenter, NewsEmbedding, NewsNERTagger, 
    MorphVocab, Doc
)  #

# --- LANGCHAIN: ЯДРО И ОБРАБОТКА ТЕКСТА ---
from langchain_core.documents import Document  #
from langchain_core.prompts import ChatPromptTemplate  #
from langchain_text_splitters import RecursiveCharacterTextSplitter  #

# --- LANGCHAIN: ВЕКТОРНЫЕ БАЗЫ И EMBEDDINGS ---
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  #
from langchain_chroma import Chroma  #

# --- LANGCHAIN: ЗАГРУЗЧИКИ И ЦЕПОЧКИ ---
from langchain_community.document_loaders import (
    DirectoryLoader, 
    UnstructuredWordDocumentLoader
)  #
from langchain_classic.chains.combine_documents import create_stuff_documents_chain  #
from langchain_classic.chains import create_retrieval_chain  #

# --- RETRIEVERS И ПЕРЕРАНЖИРОВАНИЕ ---
from langchain_community.retrievers import BM25Retriever  #
from langchain_classic.retrievers import EnsembleRetriever  #
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever  #
from langchain_cohere import CohereRerank  #
import cohere  #

print("✅ RAG-библиотеки успешно импортированы")

In [ ]:
# Сканирование директорий (File Discovery) - поиск и фильтрация поддерживаемых форматов (.docx, .txt, .xlsx)
knowledge_base_path = r"####################" # ввЕСТИ ПОЛНЫЙ ПУТЬ ДИРЕКТОРИИ
SUPPORTED_EXTENSIONS = ["*.txt", "*.docx", "*.xls", "*.xlsx"]

knowledge_path = Path(knowledge_base_path)
all_files = []
for ext in SUPPORTED_EXTENSIONS:
    all_files.extend(knowledge_path.rglob(ext))

print(f"✅ Найдено файлов: {len(all_files)}")
for f in all_files:
    print(f"   └─ {f.name}  [{f.suffix}]")

In [ ]:
# Логика парсинга: классификация документов и Regex/NLP экстракторы метаданных
# Покрываемые типы:
#   lease_contract        — Договор аренды (.docx)
#   agreement_termination — Соглашение о расторжении (.doc/.docx)
#   receipt               — Квитанции с несколькими плательщиками (.docx)
#   act_excel             — Акт об оплате (.xls/.xlsx)
#   invoice_excel         — Счёт на оплату (.xls/.xlsx)
#   invoice_docx          — Счёт на оплату в виде таблицы Word (.docx)
#   utility_invoice       — Коммунальный счёт без имён (.docx)

segmenter   = Segmenter()
morph_vocab = MorphVocab()
emb         = NewsEmbedding()
ner_tagger  = NewsNERTagger(emb)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("OK: Все библиотеки и Natasha загружены")


#Функция-фильтр которая проверяет является ли найденное имя реальным арендатором или мусором.
def is_valid_tenant(name: str) -> bool:
    nl = name.lower().strip()
    my_names   = ['Ввести имя арендодателя', 'ставропольское', 'сбербанк'] #Ввести имена самого арендодателя (владельца недвижимости). 
                                                                           #Если в тексте нашли — это не арендатор, это хозяин, отсекаем.
    stop_words = ['арендодатель', 'арендатор', 'оуфмс', 'мвд', 'ифнс',     
                  'отделом', 'краю', 'помещение', 'сторона', 'действующий',#Cлужебные слова которые парсер мог ошибочно принять за имя.
                  'гладских', 'директор', 'бухгалтер', 'представитель'] # "Арендатор", "директор", "представитель" — это роли и должности, 
    if any(me in nl for me in my_names):   return False                 # А не имена людей или организаций.
    if any(sw in nl for sw in stop_words): return False
    # Реквизиты не являются именами: ОГРН, ОГНРИП, ИНН
    if re.match(r'^(огрн|огнрип|огр)', nl) or re.match(r'^инн', nl): return False
    if len(nl) < 4: return False
    return True


def clean_org_name(raw: str) -> str:
    """Убирает ИНН/адрес/КПП из конца названия, нормализует пробелы и кавычки."""
    s = re.sub(r'\s+(?:ИНН|адрес|КПП|тел|ОГРН|оф\.).*', '', raw)
    s = re.sub(r'\s+', ' ', s).strip().rstrip(',').rstrip('»\"')
    return s


# ОПРЕДЕЛЕНИЕ ТИПА ДОКУМЕНТА
# Порядок: имя файла (точно) → текст (приблизительно) → дефолт


MONTH_WORDS = ['январ', 'феврал', 'март', 'апрел', 'май', 'июн', 'июл',
               'август', 'сентябр', 'октябр', 'ноябр', 'декабр', 'декбрь']

def detect_doc_type(file_path: Path, text_preview: str = '') -> str:
    name = file_path.stem.lower()
    ext  = file_path.suffix.lower()
    tl   = text_preview[:400].lower()

    # --- По имени файла ---
    if any(x in name for x in ['расторж', 'разрыв', 'соглашен']):
        return 'agreement_termination'
    if 'квитанц' in name:
        return 'receipt'
    if 'коммунал' in name and ext == '.docx':
        return 'utility_invoice'
    if any(x in name for x in ['счет', 'счёт']) and ext == '.docx':
        return 'invoice_docx'
    if 'акт' in name and ext in ('.xls', '.xlsx'):
        return 'act_excel'
    if any(x in name for x in ['счет', 'счёт']) and ext in ('.xls', '.xlsx'):
        return 'invoice_excel'
    if any(x in name for x in ['договор', 'contract']):
        return 'lease_contract'

    # --- По тексту ---
    if 'расторж' in tl or ('соглашение' in tl and 'арендатор' in tl):
        return 'agreement_termination'
    if 'плательщик' in tl:
        return 'receipt'
    if ext in ('.xls', '.xlsx'):
        if 'заказчик' in tl: return 'act_excel'
        if 'покупатель' in tl: return 'invoice_excel'
        return 'act_excel'

    return 'lease_contract'  # дефолт для docx


# ЧТЕНИЕ ФАЙЛОВ
# Гарантия: всегда возвращает строку, никогда не None.


def read_file(file_path: Path) -> str:
    ext = file_path.suffix.lower()
    try:
        if ext == '.txt':
            raw      = file_path.read_bytes()
            encoding = chardet.detect(raw)['encoding'] or 'utf-8'
            return raw.decode(encoding, errors='replace')

        elif ext == '.docx':
            doc = DocxDocument(file_path)
            paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
            if paragraphs:
                return '\n'.join(paragraphs)
            # Fallback на таблицы (Word-шаблоны типа счетов)
            rows = []
            for table in doc.tables:
                for row in table.rows:
                    cells = [c.text.strip() for c in row.cells if c.text.strip()]
                    deduped = []
                    for c in cells:
                        if not deduped or c != deduped[-1]:
                            deduped.append(c)
                    if deduped:
                        rows.append(' | '.join(deduped))
            return '\n'.join(rows)

        elif ext in ('.xls', '.xlsx'):
            engine = 'xlrd' if ext == '.xls' else 'openpyxl'
            sheets = pd.read_excel(file_path, sheet_name=None, engine=engine, header=None)
            parts  = []
            for sheet_name, df in sheets.items():
                df = df.fillna('')
                rows = []
                for _, row in df.iterrows():
                    vals = [str(v).strip() for v in row.values
                            if str(v).strip() and str(v).strip() != 'nan']
                    if vals:
                        rows.append(' | '.join(vals))
                if rows:
                    parts.append('[Лист: ' + sheet_name + ']\n' + '\n'.join(rows))
            return '\n\n'.join(parts)

    except Exception as e:
        logger.warning("Ошибка чтения '{}': {}".format(file_path.name, e))

    return ''



# ПАРСЕРЫ ПО ТИПУ ДОКУМЕНТА


def _parse_lease_or_termination(text: str) -> dict:
    """
    Договор аренды и Соглашение о расторжении — одинаковая структура шапки.
    Логика: берём 450 символов перед ПОСЛЕДНИМ якорем «именуемый в дальнейшем Арендатор».
    Покрывает:
      A. ООО/АО/КПК  — через «с одной стороны [и] ORG_TYPE NAME»
      B. ИП           — ORG_TYPE = «Индивидуальный предприниматель», + tenant_person
      C. Физлицо       — ФИО + ИНН/Паспорт/именуемый
    """
    result = {}
    header = text[:1200]
    anchors = list(re.finditer(
        r'именуем\w+\s+в\s+дальнейшем\s+[«\"\']?Арендатор',
        header, re.IGNORECASE
    ))
    if not anchors:
        return result

    anchor_pos    = anchors[-1].start()
    before_anchor = header[max(0, anchor_pos - 450): anchor_pos]

    # Паттерн A + B: юрлицо или ИП
    org_m = re.search(
        r'с одной стороны\s+(?:и\s+)?'
        r'(ООО|ИП|АО|ЗАО|ОАО|КПК|ПАО|НКО|Индивидуальный предприниматель)\s+'
        r'([«\"»]?[А-ЯЁа-яё\w\s«»\"]{2,80}?[»\"»]?)'
        r'(?:,|\s+адрес|\s+ИНН|\s+ОГРН|\s+именуем|\s{2,})',
        before_anchor, re.IGNORECASE | re.DOTALL
    )
    if org_m:
        org_type = org_m.group(1).strip()
        org_name = clean_org_name(org_m.group(2))
        is_ip    = org_type.lower() in ('ип', 'индивидуальный предприниматель')
        full     = re.sub(r'\s+', ' ', org_type + ' ' + org_name).strip()
        result['tenant_org']  = full
        result['tenant']      = full
        result['tenant_type'] = 'org'
        if is_ip:
            result['tenant_person'] = org_name
        return result

    # Паттерн C: физлицо — ФИО + ИНН / Паспорт / именуемый
    per_m = re.search(
        r'([А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+)'
        r'(?:,\s*ИНН|\s+ИНН|\s+Паспорт|\s+паспорт|\s+именуем|\s{2,})',
        before_anchor
    )
    if per_m and is_valid_tenant(per_m.group(1)):
        result['tenant_person'] = per_m.group(1).strip()
        result['tenant']        = result['tenant_person']
        result['tenant_type']   = 'person'

    return result


def _parse_receipt(text: str) -> dict:
    """
    Квитанции — один файл, несколько плательщиков.
    tenant = строка через запятую, tenant_list = список для фильтрации.
    """
    tenants = re.findall(r'Плательщик[:\s]+([^\n]+)', text)
    tenants = [t.strip() for t in tenants if t.strip()]
    periods = re.findall(r'Период оплаты\s+([^\n]+)', text)
    amounts = re.findall(r'К оплате\s+([\d]+)', text)
    return {
        'tenant':       ', '.join(tenants) if tenants else 'НЕ ОПРЕДЕЛЁН',
        'tenant_list':  tenants,
        'tenant_type':  'mixed',
        'periods':      [p.strip() for p in periods],
        'amounts':      [a.strip() for a in amounts],
    }


def _parse_excel_doc(flat: str) -> dict:
    """
    Акт и Счёт в формате Excel.
    Заказчик (Акт) / Покупатель (Счёт) → tenant_org.
    """
    result = {}
    for marker in ['Заказчик', 'Покупатель']:
        m = re.search(
            marker + r'\s*:?\s*\|?\s*((?:ООО|ИП|КПК|АО|ЗАО|ПАО|НКО)[^\n\|]{3,100})',
            flat
        )
        if m:
            name = clean_org_name(m.group(1)).strip('«»\"')
            if is_valid_tenant(name):
                result['tenant']      = name
                result['tenant_org']  = name
                result['tenant_type'] = 'org'
            break

    # Сумма
    summa = re.search(r'Всего к оплате:\s*\|?\s*([\d\s]+)', flat)
    if summa:
        result['amount'] = summa.group(1).strip()

    # Период из строки описания
    period_m = re.search(
        r'за\s+((?:аренд\w*|коммунальн\w*)\s+(?:плат\w+\s+)?(?:в\s+)?\w+\s+\d{4})',
        flat, re.IGNORECASE
    )
    if period_m:
        result['period'] = period_m.group(1).strip()

    # Номер и дата документа
    doc_num_m = re.search(
        r'(?:Акт|Счет|Счёт)\s*(?:на\s+оплату\s+)?№\s*(\S+)\s+от\s+([\d\w\s]+г\.?)',
        flat, re.IGNORECASE
    )
    if doc_num_m:
        result['doc_number'] = doc_num_m.group(1)
        result['doc_date']   = doc_num_m.group(2).strip()

    return result


def _parse_invoice_docx(text: str, filename_stem: str) -> dict:
    """
    Счёт в виде таблицы Word (docx).
    Имя покупателя часто в имени файла (.......).
    Список офисов — из строк таблицы.
    """
    result = {}

    # Покупатель / Плательщик — имя может быть на той же строке или на следующей
    for marker in ['Покупатель', 'Плательщик']:
        # Вариант A: имя на следующей строке (напр. "Покупатель (Заказчик):\nООО «.....»")
        m = re.search(
            marker + r'(?:\s*\([^)]+\))?\s*:?\s*\n+\s*'
            r'((?:ООО|ИП|КПК|АО|ЗАО|ПАО|НКО|Индивидуальный предприниматель)\s+[^\n]{2,80})',
            text
        )
        # Вариант B: имя на той же строке (напр. "Покупатель: ООО «.......»")
        if not m:
            m = re.search(
                marker + r'[^:\n]*:\s*'
                r'((?:ООО|ИП|КПК|АО|ЗАО|ПАО|НКО|Индивидуальный предприниматель)\s+[^\n]{2,80})',
                text
            )
        if m:
            val = clean_org_name(m.group(1)).strip('«»\"')
            if is_valid_tenant(val):
                result['tenant']      = val
                result['tenant_org']  = val
                result['tenant_type'] = 'org'
                break

    # Fallback: имя из имени файла (...., ...... и т.д.)
    if 'tenant' not in result:
        STOP = {'счет','счёт','оплату','аренда','аренды','мира','коммун','офис',
                'январ','феврал','март','апрел','май','июн','июл','август',
                'сентябр','октябр','ноябр','декабр','декбрь','нояб','октяб',
                'от','год','лет','акт','договор'}
        parts = [p.lower() for p in filename_stem.split('_') if len(p) > 3 and not p.isdigit()]
        candidates = [p for p in parts if not any(sw in p for sw in STOP)]
        if candidates:
            result['tenant']      = candidates[-1].capitalize()
            result['tenant_type'] = 'person'

    # Офисы
    offices = re.findall(r'офис\s*(?:№\s*)?(\d+)', text, re.IGNORECASE)
    if offices:
        result['offices'] = sorted(set(offices), key=int)

    # Сумма и дата
    summa = re.search(r'(?:Всего к оплате|Итого)\s*:?\s*\|?\s*([\d\s,]+)', text)
    if summa:
        result['amount'] = summa.group(1).strip()

    doc_num_m = re.search(
        r'(?:Счет|Счёт)\s+(?:на\s+оплату\s+)?№\s*(\S+)\s+от\s+([\d\w\s]+г\.?)',
        text, re.IGNORECASE
    )
    if doc_num_m:
        result['doc_number'] = doc_num_m.group(1)
        result['doc_date']   = doc_num_m.group(2).strip()

    return result


def _parse_utility_invoice(text: str) -> dict:
    """
    Коммунальный счёт — нет имён арендаторов, только офисы.
    tenant = НЕ ОПРЕДЕЛЁН (имена подтянутся из договоров по офису).
    """
    offices = re.findall(r'офис\s*(?:№\s*)?(\d+)', text, re.IGNORECASE)
    result = {
        'tenant':      'НЕ ОПРЕДЕЛЁН',
        'tenant_type': None,
    }
    if offices:
        result['offices'] = sorted(set(offices), key=int)

    summa = re.search(r'(?:Всего к оплате|Итого)\s*:?\s*\|?\s*([\d\s,]+)', text)
    if summa:
        result['amount'] = summa.group(1).strip()

    return result


# ---------------------------------------------------------------------------
# NATASHA — Уровень 2 (только если RegEx не нашёл)
# ИСПРАВЛЕНИЕ: приоритет ORG над PER — устраняет проблему «Евгения Викторовича»
# ---------------------------------------------------------------------------

def extract_with_natasha(text: str) -> dict:
    result = {}
    try:
        preamble   = text[:3500]
        doc_nat    = Doc(preamble)
        doc_nat.segment(segmenter)
        doc_nat.tag_ner(ner_tagger)
        text_lower = preamble.lower()

        anchors = list(re.finditer(r'дальнейшем\s*[«\" ]*арендатор', text_lower))
        if not anchors:
            return result

        anchor_pos = anchors[-1].start()
        org_candidates = []
        per_candidates = []

        for span in doc_nat.spans:
            if span.type not in ('ORG', 'PER'):
                continue
            name = span.text.strip()
            if not is_valid_tenant(name):
                continue
            # Реквизиты типа "ОГРН №232432542354356" Natasha иногда помечает как ORG
            if re.search(r'\d{8,}', name):  # длинные числа = реквизит, не имя
                continue
            dist = anchor_pos - span.stop
            if 0 < dist < 400:
                if span.type == 'ORG':
                    org_candidates.append((name, dist))
                else:
                    per_candidates.append((name, dist))

        # ORG приоритет над PER — исправляет «Ивана Викторовича»
        if org_candidates:
            org_candidates.sort(key=lambda x: x[1])
            best_name = org_candidates[0][0]
            result['tenant_nlp']  = best_name
            result['nlp_type']    = 'ORG'
            result['nlp_all']     = [c[0] for c in org_candidates] + [c[0] for c in per_candidates]
        elif per_candidates:
            per_candidates.sort(key=lambda x: x[1])
            best_name = per_candidates[0][0]
            result['tenant_nlp'] = best_name
            result['nlp_type']   = 'PER'
            result['nlp_all']    = [c[0] for c in per_candidates]

    except Exception as e:
        logger.error("Ошибка Natasha: {}".format(e))

    return result



# FALLBACK: имя файла


def extract_from_filename(file_path: Path) -> dict:
    name  = file_path.stem
    STOP = {'счет','счёт','оплату','аренда','аренды','мира','коммун','офис',
            'январ','феврал','март','апрел','май','июн','июл','август',
            'сентябр','октябр','ноябр','декабр','декбрь','нояб','октяб',
            'от','год','лет','акт','договор','соглашен','расторж'}

    org_m = re.search(r'(ООО|ИП|АО)[\s_-]*(\w+)', name, re.IGNORECASE)
    if org_m:
        return {
            'tenant':      org_m.group(1) + ' ' + org_m.group(2),
            'tenant_type': 'org',
        }
    parts = [p.lower() for p in name.split('_') if len(p) > 3 and not p.isdigit()]
    candidates = [p for p in parts if not any(sw in p for sw in STOP)]
    if candidates:
        return {
            'tenant':      candidates[-1].capitalize(),
            'tenant_type': 'person',
        }
    return {}



# ГЛАВНЫЙ ПАЙПЛАЙН: extract_metadata
# Маршрут:  detect_doc_type → нужный парсер → Natasha (если пусто) → filename → НЕ ОПРЕДЕЛЁН


def extract_metadata(file_path: Path, text: str = '', use_nlp: bool = False) -> dict:

    # 1. Файловая система
    parts   = file_path.parts
    address = 'unknown'
    office  = 'unknown'
    
    for part in parts[:-1]:  # [:-1] — пропускаем последний элемент (имя файла)
        pl = part.lower()
        if any(x in pl for x in ['мира', 'ленина', 'победы']): address = part
        if any(x in pl for x in ['офис', 'магазин', 'office']): office = part
            
    # Fallback: адрес из имени файла если в папках не нашли
    if address == 'unknown':
        fname = file_path.stem.lower()
        if '284' in fname:               #ВВести свои адреса для фильтрации
            address = '>>>>>>>> >>>>>>>>'#ВВести свои адреса для фильтрации
        elif '144' in fname:
            address = '>>>>>>>> >>>>>>>>'#ВВести свои адреса для фильтрации
            
    ext      = file_path.suffix.lower()
    doc_type = detect_doc_type(file_path, text)

    metadata = {
        'source':            str(file_path),
        'filename':          file_path.name,
        'doc_type':          doc_type,
        'address':           address,
        'office':            office,
        'folder':            file_path.parent.name,
        'full_path':         ' | '.join(parts),
        'extraction_method': 'fs',
        # Гарантированные поля
        'tenant':            'НЕ ОПРЕДЕЛЁН',
        'tenant_org':        None,
        'tenant_person':     None,
        'tenant_type':       None,
    }

    year_m = re.search(r'(20\d{2})', file_path.name)
    if year_m:
        metadata['year'] = int(year_m.group(1))

    # 2. Парсер по типу документа
    parsed = {}
    if text:
        if doc_type in ('lease_contract', 'agreement_termination'):
            parsed = _parse_lease_or_termination(text)
            metadata['extraction_method'] = 'fs + regex'

        elif doc_type == 'receipt':
            parsed = _parse_receipt(text)
            metadata['extraction_method'] = 'fs + regex'

        elif doc_type in ('act_excel', 'invoice_excel'):
            parsed = _parse_excel_doc(text)
            metadata['extraction_method'] = 'fs + regex'

        elif doc_type == 'invoice_docx':
            parsed = _parse_invoice_docx(text, file_path.stem)
            metadata['extraction_method'] = 'fs + regex'

        elif doc_type == 'utility_invoice':
            parsed = _parse_utility_invoice(text)
            metadata['extraction_method'] = 'fs + regex'

    metadata.update(parsed)

    # 3. Natasha — только если tenant не найден и это текстовый документ
    if (use_nlp
            and metadata.get('tenant') in (None, 'НЕ ОПРЕДЕЛЁН')
            and doc_type in ('lease_contract', 'agreement_termination')
            and text):
        nlp_data = extract_with_natasha(text)
        if 'tenant_nlp' in nlp_data:
            t    = nlp_data['tenant_nlp']
            ttyp = nlp_data.get('nlp_type', '')
            metadata['tenant']  = t
            metadata['nlp_all'] = nlp_data.get('nlp_all', [])
            if ttyp == 'PER':
                metadata['tenant_person'] = t
                metadata['tenant_type']   = 'person'
            else:
                metadata['tenant_org']  = t
                metadata['tenant_type'] = 'org'
            metadata['extraction_method'] = 'fs + regex + nlp'
        else:
            metadata['extraction_method'] = 'fs + regex + nlp_failed'

    # 4. Fallback: имя файла
    if metadata.get('tenant') in (None, 'НЕ ОПРЕДЕЛЁН') and doc_type != 'utility_invoice':
        fname_data = extract_from_filename(file_path)
        if fname_data.get('tenant'):
            metadata.update(fname_data)
            metadata['extraction_method'] = metadata.get('extraction_method', 'fs') + ' + filename'

    # 5. Финальная гарантия
    if not metadata.get('tenant'):
        metadata['tenant'] = 'НЕ ОПРЕДЕЛЁН'

    return metadata


print("OK: extract_metadata v3 загружен")
print("    Типы: lease_contract, agreement_termination, receipt,")
print("          act_excel, invoice_excel, invoice_docx, utility_invoice")
print("    Natasha: ORG приоритет над PER (fix: Евгения Викторовича)")
print("    Гарантия: каждый файл -> статус")


In [ ]:
# ЦИКЛ ОБРАБОТКИ ДОКУМЕНТОВ + ЗАПОЛНЕНИЕ SQLite 

try:
    cursor.close()
except:
    pass
try:
    conn.close()
except:
    pass

DB_PATH = "knowledge.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# ── 1. Создаём базу и таблицы заново
conn = sqlite3.connect(DB_PATH, timeout=10)
cursor = conn.cursor()

cursor.executescript("""
    CREATE TABLE IF NOT EXISTS documents (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        filename          TEXT,
        source            TEXT UNIQUE,
        doc_type          TEXT,
        address           TEXT,
        office            TEXT,
        tenant            TEXT,
        tenant_type       TEXT,
        amount            TEXT,
        period            TEXT,
        year              INTEGER,
        doc_number        TEXT,
        doc_date          TEXT,
        extraction_method TEXT
    );

    CREATE TABLE IF NOT EXISTS receipt_entries (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        source  TEXT,
        tenant  TEXT,
        amount  TEXT,
        period  TEXT
    );
""")
conn.commit()
print(f"🗄️  База {DB_PATH} создана, начинаем заполнение...")

# ── 2. Основной цикл 

documents = []

for file_path in tqdm(all_files, desc="Анализ документов"):
    content = read_file(file_path)
    if not content.strip():
        continue

    metadata = extract_metadata(file_path, text=content, use_nlp=True)

    # Формируем Document для Chroma — без изменений
    tenant_info = metadata.get('tenant', 'НЕ ОПРЕДЕЛЕН')
    enhanced_content = (
        f"ОБЪЕКТ: {metadata['address']} | ОФИС: {metadata['office']} | "
        f"АРЕНДАТОР: {tenant_info} | ГОД: {metadata.get('year', 'не указан')}\n"
        f"ФАЙЛ: {metadata['filename']}\n"
        f"СОДЕРЖАНИЕ:\n{content}"
    )

    doc = Document(page_content=enhanced_content, metadata=metadata)
    documents.append(doc)

    # INSERT в таблицу documents
    cursor.execute("""
        INSERT OR IGNORE INTO documents
            (filename, source, doc_type, address, office,
             tenant, tenant_type, amount, period, year,
             doc_number, doc_date, extraction_method)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        metadata.get('filename'),
        metadata.get('source'),
        metadata.get('doc_type'),
        metadata.get('address'),
        metadata.get('office'),
        metadata.get('tenant'),
        metadata.get('tenant_type'),
        metadata.get('amount'),
        (metadata.get('periods') or [None])[0] if isinstance(metadata.get('periods'), list)
            else metadata.get('period'),
        metadata.get('year'),
        metadata.get('doc_number'),
        metadata.get('doc_date'),
        metadata.get('extraction_method'),
    ))

    # Квитанции: каждый плательщик — отдельная строка в receipt_entries
    if metadata.get('doc_type') == 'receipt':
        tenant_list = metadata.get('tenant_list', [])
        amounts     = metadata.get('amounts', [])
        periods     = metadata.get('periods', [])
        for t, a, p in zip_longest(tenant_list, amounts, periods):
            if t:
                cursor.execute("""
                    INSERT INTO receipt_entries (source, tenant, amount, period)
                    VALUES (?, ?, ?, ?)
                """, (metadata.get('source'), t, a, p))

    # Вывод в консоль — без изменений
    method = metadata.get('extraction_method', 'fs')
    print(f"\n📄 Файл: {file_path.name}")
    print(f"   ├─ Путь: {metadata['address']} / {metadata['office']}")
    if "nlp" in method:
        raw_orgs = metadata.get('nlp_all', [])
        print(f"   ├─ ✨ [NLP Natasha] Нашла в тексте: {raw_orgs}")
        print(f"   └─ 🎯 Итоговый Арендатор: {tenant_info}")
    else:
        print(f"   └─ ⚡ [RegEx] Арендатор найден быстро: {tenant_info}")

# ── 3. Сохраняем — conn оставляем открытым для Ячейки 10 (Router) 
conn.commit()

# ── 5. Итог 
total_docs  = cursor.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
total_rcpts = cursor.execute("SELECT COUNT(*) FROM receipt_entries").fetchone()[0]

print(f"\n{'='*60}")
print(f"✅ Обработка завершена! Всего в базе: {len(documents)} чанков.")
print(f"🗄️  SQLite → documents: {total_docs} строк | receipt_entries: {total_rcpts} строк")

print(f"\n📋 Проверка — найденные арендаторы:")
rows = cursor.execute("""
    SELECT address, office, tenant, doc_type
    FROM documents
    WHERE tenant != 'НЕ ОПРЕДЕЛЁН'
    ORDER BY address, office
""").fetchall()
for row in rows:
    print(f"   {row[0]:<15} | {row[1]:<12} | {row[2]:<30} | {row[3]}")

In [ ]:
# Токенизация: расчет общего объема базы знаний для модели gpt-5-nano
#Сбор всех файлов в одну гигантскую текстовую строку
#Расчет реального количества токенов использованных в документах

MODEL = "gpt-5-nano" #Выбор «линейку», которой будешь мерить
entire_knowledge_base = "\n\n".join(doc.page_content for doc in documents)

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
print(f"Total tokens for {MODEL}: {len(tokens):,}")

In [ ]:
# # Дифференцированный чанкинг: разделение текстов на блоки и сохранение таблиц целиком
#Разделение по типам: Код отделяет файлы Excel от текстовых документов (.docx, .txt). 


excel_docs = [d for d in documents if d.metadata["doc_type"] == "excel"]
text_docs  = [d for d in documents if d.metadata["doc_type"] != "excel"]

# Инициализируем энкодер
encoding = tiktoken.get_encoding("cl100k_base")

# Диагностика — смотрим распределение токенов
token_counts = [
    len(encoding.encode(doc.page_content)) for doc in text_docs
]
print(f"Мин токенов:    {min(token_counts)}")
print(f"Макс токенов:   {max(token_counts)}")
print(f"Среднее:        {int(sum(token_counts)/len(token_counts))}")
print(f"До 600 токенов: {sum(1 for t in token_counts if t <= 600)} документов")
print(f"Более 600:      {sum(1 for t in token_counts if t > 600)} документов")

# Разделяем — короткие целиком, длинные режем
CHUNK_THRESHOLD = 600

short_docs = []
long_docs = []

for doc in text_docs:
    token_count = len(encoding.encode(doc.page_content))
    if token_count <= CHUNK_THRESHOLD:
        short_docs.append(doc)
    else:
        long_docs.append(doc)

# Длинные режем - устанавливаем перекрытия символов
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=400,
    separators=["\n\n", "\n", ".", " ", ""]
)
split_texts = text_splitter.split_documents(long_docs)

# Собираем всё вместе
chunks = split_texts + short_docs + excel_docs

print(f"\nКоротких документов (целиком): {len(short_docs)}")
print(f"Длинных документов (порезаны): {len(long_docs)} -> {len(split_texts)} чанков")
print(f"Excel (целиком):               {len(excel_docs)}")
print(f"Итого чанков:                  {len(chunks)}")

In [ ]:
# ChromaDB: создание векторного индекса и добавление новых документов с дедупликацией

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db_name = "vector_db_v2"

if os.path.exists(db_name):
    print(f"База '{db_name}' найдена — подключаемся...")
    vectorstore = Chroma(
        persist_directory=db_name,
        embedding_function=embeddings
    )
else:
    print("База не найдена — создаём с нуля...")
    vectorstore = Chroma(
        persist_directory=db_name,
        embedding_function=embeddings
    )

existing_data = vectorstore.get()
existing_sources = set()
for meta in existing_data["metadatas"]:
    if meta and "source" in meta:
        existing_sources.add(meta["source"])

print(f"Уже в базе: {len(existing_sources)} уникальных файлов / {len(existing_data['ids'])} чанков")

new_chunks = [
    chunk for chunk in chunks
    if chunk.metadata.get("source") not in existing_sources
]

if not new_chunks:
    print("Новых файлов нет. База актуальна!")
else:
    print(f"Новых чанков для добавления: {len(new_chunks)}")
    
    new_ids = []
    counter = {}
    for chunk in new_chunks:
        # --- НОВЫЙ БЛОК ОЧИСТКИ ---
        # Проверяем метаданные: если там есть пустой список (как твой 'amounts': []), 
        # ChromaDB выдаст ошибку. Мы просто удаляем такие ключи.
        bad_keys = [k for k, v in chunk.metadata.items() if isinstance(v, list) and len(v) == 0]
        for k in bad_keys:
            del chunk.metadata[k]
        # -------------------------

        src = chunk.metadata.get("source", "unknown")
        counter[src] = counter.get(src, 0) + 1
        uid = hashlib.md5(f"{src}_{counter[src]}".encode()).hexdigest()
        new_ids.append(uid)
    
    # Теперь Chroma получит "чистые" данные без пустых списков
    vectorstore.add_documents(documents=new_chunks, ids=new_ids)
    print(f"Добавлено {len(new_chunks)} новых чанков!")

print(f"Всего векторов в базе: {len(vectorstore.get()['ids'])}")

In [ ]:
# Ядро системы: Router (выбор пути SQL/RAG), SQL-интерфейс и гибридный Ensemble-поиск


# Cohere reranker — клиент инициализирован, но не активен (trial ключ не поддерживает rerank API)
# Когда получите платный ключ — раскомментировать CohereRerank в get_retriever()
cohere_client = cohere.Client(os.getenv("COHERE_API_KEY"))
print("✅ Cohere reranker загружен")

llm = ChatOpenAI(model="gpt-5-nano", temperature=0, timeout=120)

system_prompt = """Ты — экспертный ИИ-помощник по управлению арендной недвижимостью.

В твоей базе знаний:
- Договоры аренды (docx)
- Акты приёма-передачи (docx)
- Счета и акты об оплате (xlsx/xls)
- Коммунальные платежи (txt/xlsx)

Правила работы:
1. Отвечай ТОЛЬКО на основе предоставленного контекста
2. Если нужно посчитать сумму — складывай числа и показывай расчёт
3. Всегда указывай из какого файла взята информация
4. Если данных недостаточно — скажи какой именно информации не хватает
5. Не придумывай ничего от себя
6. Отвечай на русском языке
7. Если в контексте несколько документов с разными датами для одного объекта,
   приоритизируй самый свежий по дате в названии файла
8. Если вопрос содержит 'все', 'список', 'перечисли', 'кто' — пройдись по КАЖДОМУ
   документу в контексте и перечисли все уникальные значения
9. Если в разных документах данные противоречат друг другу — укажи это явно
10. НИКОГДА не используй данные из документов с адресом отличным от адреса в вопросе

Контекст:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


def get_retriever(query: str, vectorstore, documents):
    address_filter = None
    query_lower = query.lower()
    if '>>>>>>>>' in query_lower:
        address_filter = '>>>>>>>> >>>>>>>>'
    elif '>>>>>>>>' in query_lower:
        address_filter = '>>>>>>>> >>>>>>>>'

    # НОВОЕ: фильтр по арендатору если имя упомянуто в запросе
    tenant_name = _extract_name(query)

    if address_filter and tenant_name:
        name_words = tenant_name.split()
        filtered_docs = [
            doc for doc in documents
            if doc.metadata.get('address') == address_filter
            and any(
                word in doc.metadata.get('tenant', '').lower()
                for word in name_words
            )
        ]
        # Если по арендатору ничего не нашли — берём все по адресу
        if not filtered_docs:
            filtered_docs = [
                doc for doc in documents
                if doc.metadata.get('address') == address_filter
            ]
        print(f"  Фильтр: '{address_filter}' + '{tenant_name}': {len(filtered_docs)}/{len(documents)} документов")

    elif address_filter:
        filtered_docs = [
            doc for doc in documents
            if doc.metadata.get('address') == address_filter
        ]
        print(f"  Фильтр по адресу '{address_filter}': {len(filtered_docs)}/{len(documents)} документов")

    elif tenant_name:
        # Ищем документы где tenant содержит ЛЮБОЕ из слов запроса
        name_words = tenant_name.split()
        filtered_docs = [
            doc for doc in documents
            if any(
                word in doc.metadata.get('tenant', '').lower()
                for word in name_words
            )
        ]
        if not filtered_docs:
            filtered_docs = documents
        print(f"  Фильтр по арендатору '{tenant_name}': {len(filtered_docs)}/{len(documents)} документов")
        for d in filtered_docs[:5]:
            print(f"    → {d.metadata.get('filename')} | tenant: {d.metadata.get('tenant')}")
    else:
        filtered_docs = documents

    aggregate_keywords = ['все', 'список', 'перечисли', 'кто', 'сколько арендаторов']
    is_aggregate = any(kw in query_lower for kw in aggregate_keywords)
    k = 6 if is_aggregate else 4  # увеличили с 3 до 4
    print(f"  Режим: {'агрегирующий' if is_aggregate else 'точный'} → k={k}")

    bm25_retriever = BM25Retriever.from_documents(filtered_docs)
    bm25_retriever.k = k

    # Если фильтр только по арендатору — используем только BM25
    # Vector retriever не умеет фильтровать по tenant и тянет чужие документы
    if tenant_name and not address_filter:
        return bm25_retriever

    search_kwargs = {"k": k}
    if address_filter:
        search_kwargs["filter"] = {"address": address_filter}

    vector_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs=search_kwargs
    )

    return EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.4, 0.6]
    )


def _extract_name(query: str) -> str:
    """Извлекает имя арендатора из запроса, убирая стоп-слова."""
    stop = {
        'сколько', 'платит', 'арендует', 'договор', 'документы',
        'покажи', 'найди', 'все', 'список', 'аренда', 'арендная',
        'плата', 'платёж', 'мира', '144', '284', 'офис', 'какой',
        'какая', 'размер', 'сумма', 'счет', 'акт', 'есть', 'какие',
        'условия', 'расторжения', 'файлы', 'там', 'площадь',  # ← добавить
        'помещения', 'помещение', 'какова', 'укажи', 'скажи'   # ← добавить
    }
    words = [w for w in query.lower().split() if w not in stop and len(w) > 2]
    # Возвращаем ВСЕ слова склеенные для поиска, а не только первое
    return ' '.join(words) if words else ''


def _run_sql(query: str) -> str:
    """
    Шаблонные SQL-запросы к SQLite.
    Возвращает строку с результатом или 'FALLBACK_TO_RAG' если данных нет.
    cursor — объект из Ячейки 5, остаётся открытым.
    """
    q = query.lower()
    name = _extract_name(query)

    try:
        # Шаблон 1: список арендаторов
        if any(w in q for w in ['кто арендует', 'все арендаторы', 'список арендаторов', 'перечисли']): 
            address_filter = '%144%' if '144' in q else '%284%' if '284' in q else '%' #ВВести свои адреса для фильтрации
            rows = cursor.execute("""
                SELECT DISTINCT tenant, office, doc_type
                FROM documents
                WHERE tenant != 'НЕ ОПРЕДЕЛЁН'
                  AND address LIKE ?
                ORDER BY office, tenant
            """, (address_filter,)).fetchall()
            if not rows:
                return "Арендаторы не найдены."
            return "Арендаторы:\n" + "\n".join(
                f"- {r[0]} | {r[1]} | {r[2]}" for r in rows
            )

        # Шаблон 2: сколько платит — ищем суммы в актах
        if any(w in q for w in ['сколько платит', 'размер', 'сумма', 'арендная плата', 'платёж']):
            if not name:
                return "FALLBACK_TO_RAG"
            rows = cursor.execute("""
                SELECT tenant, amount, period, doc_type, filename
                FROM documents
                WHERE amount IS NOT NULL AND amount != ''
                  AND LOWER(tenant) LIKE ?
                ORDER BY year DESC
                LIMIT 5
            """, (f'%{name}%',)).fetchall()
            if rows:
                result = f"Суммы по запросу '{name}':\n"
                result += "\n".join(
                    f"- {r[0]}: {r[1]} руб. | {r[2]} | {r[4]}" for r in rows
                )
                return result
            else:
                return "FALLBACK_TO_RAG"

        # Шаблон 3: документы по арендатору
        if any(w in q for w in ['документы', 'договор', 'акты', 'счета', 'файлы']):
            if not name:
                return "FALLBACK_TO_RAG"
            rows = cursor.execute("""
                SELECT filename, doc_type, doc_date, address, office
                FROM documents
                WHERE LOWER(tenant) LIKE ?
                ORDER BY year DESC
            """, (f'%{name}%',)).fetchall()
            if not rows:
                return "Документы не найдены."
            return "Документы:\n" + "\n".join(
                f"- {r[0]} | {r[1]} | {r[2]} | {r[3]} {r[4]}" for r in rows
            )

        return "FALLBACK_TO_RAG"

    except Exception as e:
        return f"Ошибка SQL: {e}"


def ask(query: str):
    print(f"🗣️ Вопрос: {query}\n⏳ Classifying...\n")

    router_prompt = f"""Ты — классификатор запросов по арендной недвижимости.
Определи тип запроса:

"sql"  — вопрос про факты: кто арендует, сколько платит, список арендаторов,
         номера договоров, суммы, даты, перечисление, подсчёт.

"rag"  — вопрос про содержание документов: условия договора, пункты об
         ответственности, порядок расторжения, что написано в документе.

"both" — нужны и факты и содержание: например "кто арендует офис 8 и на каких условиях".

Ответь ОДНИМ словом: sql, rag или both.

Вопрос: {query}"""

    route = llm.invoke(router_prompt).content.strip().lower()
    if route not in ("sql", "rag", "both"):
        route = "rag"
    print(f"  🔀 Router → {route.upper()}\n")

    time.sleep(3)

    # Шаг 2: SQL-ветка
    sql_context = ""
    if route in ("sql", "both"):
        sql_context = _run_sql(query)

        if route == "sql":
            if sql_context == "FALLBACK_TO_RAG" or not sql_context:
                print("  ℹ️ SQL данных не нашёл → переключаемся на RAG\n")
                route = "rag"
            else:
                format_prompt = f"""Ты — помощник по аренде недвижимости.
Результат SQL-запроса по вопросу "{query}":

{sql_context}

Представь результат в читаемом виде (список или таблица).
Если результат пустой — скажи что данных не найдено.
Отвечай на русском языке."""
                answer = llm.invoke(format_prompt).content
                print("🤖 Ответ:")
                print("─" * 50)
                print(answer)
                print("─" * 50)
                print("\n📊 Источник: SQLite (структурированные данные)")
                return

    # Шаг 3: RAG-ветка
    if route in ("rag", "both"):
        retriever = get_retriever(query, vectorstore, documents)

        time.sleep(2)
        retrieved_docs = retriever.invoke(query)

        context = "\n\n".join([
            f"[{doc.metadata.get('filename', '?')}]\n{doc.page_content[:2000]}"
            for doc in retrieved_docs
        ])

        if route == "both" and sql_context and sql_context != "FALLBACK_TO_RAG":
            context = f"[SQL-данные]:\n{sql_context}\n\n" + context

        time.sleep(2)
        final_prompt = f"{system_prompt.replace('{context}', context)}\n\nВопрос: {query}"
        answer = llm.invoke(final_prompt)

        print("🤖 Ответ:")
        print("─" * 50)
        print(answer.content)
        print("─" * 50)
        print("\n📚 Источники:")
        seen = set()
        for i, doc in enumerate(retrieved_docs):
            fname  = doc.metadata.get('filename', '?')
            office = doc.metadata.get('office', '?')
            addr   = doc.metadata.get('address', '?')
            key = f"{fname}_{office}"
            if key not in seen:
                seen.add(key)
                print(f"  {i+1}. {fname} [{office} | {addr}]")


print("✅ Готово")

In [ ]:

ask("Балаян номер офиса")         #Routing SQL, 
